# Question 6 - 189. Rotate Array

Given an integer array `nums`, rotate the array to the right by `k` steps, where `k` is non-negative.

**Example 1:**

    Input: nums = [1,2,3,4,5,6,7], k = 3
    Output: [5,6,7,1,2,3,4]
    Explanation:
    rotate 1 steps to the right: [7,1,2,3,4,5,6]
    rotate 2 steps to the right: [6,7,1,2,3,4,5]
    rotate 3 steps to the right: [5,6,7,1,2,3,4]

**Example 2:**

    Input: nums = [-1,-100,3,99], k = 2
    Output: [3,99,-1,-100]
    Explanation:
    rotate 1 steps to the right: [99,-1,-100,3]
    rotate 2 steps to the right: [3,99,-1,-100]

**Constraints:**

- `1 <= nums.length <= 10^5`
- `-2^31 <= nums[i] <= 2^31 - 1`
- `0 <= k <= 10^5`

**Follow up:**

- Try to come up with as many solutions as you can. There are at least **three** different ways to solve this problem.
- Could you do it in-place with `O(1)` extra space?

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Three reversals (in place)

📘 **Revise first:** [Pattern F · reverse tricks](./0-concepts-array-and-string.ipynb) · [Part 3.8 · slicing & nums[:]](./0-concepts-array-and-string.ipynb) · [Part 7 · trap 8](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (shift by one, k times) | O(n · k) | O(1) |
| 1½ Extra array | O(n) | O(n) |
| 2️⃣ Optimized (reverse three times) | O(n) | O(1) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

"Rotate right by `k`" means: **every item moves `k` places to the right, and items that fall off the end wrap around to the front.**

```
[1, 2, 3, 4, 5, 6, 7], k = 3
 └─── first 4 ───┘ └ last 3 ┘
result: [5, 6, 7, 1, 2, 3, 4]
          ↑ the last 3 came to the front
```

So rotating right by `k` is the same as: **take the last `k` items and move that block to the front**, keeping both blocks in their original order.

**Everyday picture:** people sitting around a **round table** with numbered seats. "Everyone shift 3 seats clockwise." Nobody leaves; the people near the end just wrap around to the first seats.

**Two quick facts:**
- Rotating by `n` (the length) puts everyone back where they started. So only `k % n` matters. (`%` is the remainder: `10 % 7 = 3`.)
- The array must be changed **in place**. We can't return a new one.

### Step 0 · Clarify before coding

🗣️ *"Rotating right by k moves the last k elements to the front. I modify nums in place and return nothing. k can be larger than the array length, so I'll reduce it with k mod n."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Can `k` be bigger than `n`? | **Yes** (up to 10⁵). | Use `k % n`. Rotating by n is a no-op. |
| In place? Return value? | **In place**, return nothing. | `nums = ...` inside the function would **not** change the caller's list. Use `nums[:] = ...` or write into slots. |
| Can `k` be 0? | **Yes.** | Should leave the array untouched. |
| Size? | n and k up to 10⁵. | O(n·k) could be 10¹⁰ steps, which is too slow. |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** do exactly what the examples show: **rotate by one step, `k` times.** One step = remember the last item, shift everyone one place right, put the remembered item at the front.

🗣️ *"The literal approach rotates by one position k times. Each single rotation saves the last element, shifts everything right by one, and puts the saved element at index 0."*

**Why is it slow?** One single-step rotation touches all n items. Doing it k times is **n × k** work. With n = k = 100,000 that's 10 billion moves.

🗣️ *"That's O(n·k). With both up to 10⁵ it's far too slow. Each element moves k times, one step at a time, when it could jump straight to its final position."*

In [1]:
class SolutionBrute:
    def rotate(self, nums: list[int], k: int) -> None:
        n = len(nums)
        for _ in range(k % n):
            last = nums[-1]
            for i in range(n - 1, 0, -1):   # shift everyone right by one
                nums[i] = nums[i - 1]
            nums[0] = last

#### Step 1½ · Jump straight to the final spot (with an extra array)

Each item at index `i` ends up at index `(i + k) % n`. Write every item into a **new array** at its final position, then copy back.

🗣️ *"Every element at i lands at (i + k) mod n. If I write into a helper array, each element moves exactly once, which is O(n) time, but it costs O(n) extra memory. The follow-up asks for O(1) extra space."*

(In Python, `nums[:] = nums[-k:] + nums[:-k]` is the same idea in one line. The slices are temporary copies, so it's O(n) extra space too.)

In [2]:
class SolutionExtra:
    def rotate(self, nums: list[int], k: int) -> None:
        n = len(nums)
        out = [0] * n
        for i in range(n):
            out[(i + k) % n] = nums[i]      # each element jumps to its final seat
        nums[:] = out                       # copy back IN PLACE (not nums = out!)

### Step 2 · Optimize: the "aha" moment

We want to turn `[A-block | B-block]` into `[B-block | A-block]`, where B is the last `k` items, **without extra memory**.

> 💡 **Aha: reversing does the swap for us.**
> 1. **Reverse the whole array.** B moves to the front and A to the back, but each block is now *backwards*.
> 2. **Reverse the first `k` items.** B is the right way round again.
> 3. **Reverse the remaining items.** A is the right way round again.

```
start             [1 2 3 4 | 5 6 7]     A = 1 2 3 4,  B = 5 6 7
reverse all       [7 6 5 | 4 3 2 1]     B in front (backwards), A behind (backwards)
reverse first k   [5 6 7 | 4 3 2 1]     B fixed
reverse the rest  [5 6 7 | 1 2 3 4]     A fixed ✅
```

**Reversing in place** is easy: swap the two ends and move inward (two pointers from opposite ends, Pattern B). It needs no extra memory.

**Why it works, in one line:** reversing the whole thing swaps the **order of the blocks** but also flips the **inside of each block**. Reversing each block again un-flips the insides and leaves the new block order.

🗣️ *"Rotating right by k means moving the last k elements to the front. I can do that in place with three reversals: reverse the whole array, which puts the last k elements at the front but backwards, then reverse the first k to fix them, then reverse the remaining n − k to fix those. Each reversal is a two-pointer swap from both ends, so every element is swapped a constant number of times. O(n) time, O(1) extra space. I'll reduce k mod n first."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| Rotate one step, k times | ❌ | O(n·k). Elements creep one seat at a time. |
| Extra array, `out[(i+k)%n] = nums[i]` | ✅ simple | O(n) time and easy to explain, but O(n) extra memory. |
| Slicing `nums[:] = nums[-k:] + nums[:-k]` | ✅ Pythonic | Same as the extra array: O(n) hidden memory. Fine in practice. Mention it, but expect the follow-up. |
| `collections.deque.rotate(k)` | ⚠️ | Built-in and O(k), but converting to and from a deque costs O(n) memory, and it avoids the point of the question. |
| Cyclic replacements (follow each element's cycle) | ✅ but tricky | O(n) time and O(1) space, but you need gcd-based cycle counting. Easy to get wrong live. |
| **Three reversals** | ✅ **best** | O(n), O(1) extra, short and easy to prove. |

In [3]:
class Solution:
    def rotate(self, nums: list[int], k: int) -> None:
        n = len(nums)
        k %= n                      # rotating by n changes nothing

        def reverse(lo: int, hi: int) -> None:
            while lo < hi:          # swap ends, move inward
                nums[lo], nums[hi] = nums[hi], nums[lo]
                lo += 1
                hi -= 1

        reverse(0, n - 1)           # 1) whole array
        reverse(0, k - 1)           # 2) first k items  (the old tail)
        reverse(k, n - 1)           # 3) the rest       (the old head)

### Step 3 · Toy example walkthrough (tell it like a story)

`nums = [-1, -100, 3, 99]`, `k = 2` → expected `[3, 99, -1, -100]`

| step | operation | array |
|---|---|---|
| 0 | start (A = `-1, -100`, B = `3, 99`) | [-1, -100, **3, 99**] |
| 1 | reverse all (indexes 0..3) | [99, 3, -100, -1] |
| 2 | reverse first k = 2 (indexes 0..1) | [**3, 99**, -100, -1] |
| 3 | reverse the rest (indexes 2..3) | [3, 99, **-1, -100**] ✅ |

**Narrated:** "I want the last two, 3 and 99, at the front. Reversing everything gets them to the front but backwards: 99, 3. So I flip just those two back to 3, 99. The original front, −1 and −100, is now at the back but backwards, so I flip that part too. Done, and I never used a second array."

**`k > n` example:** `[1, 2]`, `k = 3` → `k % 2 = 1` → same as rotating by 1 → `[2, 1]`.

In [4]:
def run(cls, nums, k):
    nums = nums[:]
    cls().rotate(nums, k)
    return nums

cases = [
    (([1, 2, 3, 4, 5, 6, 7], 3), [5, 6, 7, 1, 2, 3, 4]),
    (([-1, -100, 3, 99], 2), [3, 99, -1, -100]),
    (([1, 2], 3), [2, 1]),          # k > n
    (([1], 0), [1]),
    (([1, 2, 3], 3), [1, 2, 3]),    # k == n: no change
    (([1, 2, 3], 0), [1, 2, 3]),    # k == 0
]
for args, expected in cases:
    for Impl in (SolutionBrute, SolutionExtra, Solution):
        assert run(Impl, *args) == expected, Impl.__name__
    print(args, "->", expected)

import random
for _ in range(500):
    nums = [random.randint(0, 9) for _ in range(random.randint(1, 10))]
    k = random.randint(0, 25)
    assert run(Solution, nums, k) == run(SolutionBrute, nums, k)
print("All tests passed ✅")

([1, 2, 3, 4, 5, 6, 7], 3) -> [5, 6, 7, 1, 2, 3, 4]
([-1, -100, 3, 99], 2) -> [3, 99, -1, -100]
([1, 2], 3) -> [2, 1]
([1], 0) -> [1]
([1, 2, 3], 3) -> [1, 2, 3]
([1, 2, 3], 0) -> [1, 2, 3]
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| One step, k times | O(n·k) | O(1) | Every element moves k separate times. |
| Extra array | O(n) | O(n) | Each element jumps once, into a second array. |
| **Three reversals** | **O(n)** | **O(1)** | Each element is swapped about twice in total. |

**Why O(n) time?** Reversal 1 touches all n items. Reversals 2 and 3 together touch all n items again. That's about 2n swaps, which is still "proportional to n".

**Why O(1) memory?** Swapping uses just a couple of index variables, with no second array.

**Edge cases to mention:** `k = 0`, `k = n`, `k > n` (use `k % n`), a single element.

**The Python trap (say it if you use slicing):** inside the function, `nums = nums[-k:] + nums[:-k]` only re-labels the local variable, so the caller sees **no change**. `nums[:] = ...` overwrites the contents in place. (Concepts notebook, Part 7, trap 8.)

**Likely follow-up:** *"Rotate left by k?"* → rotate right by `n − k`, or reverse the first k, then the rest, then the whole array.

---

#### 🗣️ Full interview script (about 60 seconds)

*"Rotating right by k means the last k elements move to the front. I'll first set k to k mod n, because rotating by n does nothing.*

*Brute force rotates one step k times, which is O(n·k), far too slow at 10⁵. Writing each element to index (i + k) mod n in a new array is O(n), but it uses O(n) extra space.*

*To do it in place, I use three reversals: reverse the whole array, which brings the last k elements to the front but backwards; then reverse the first k to fix their order; then reverse the remaining n − k. Each reversal is a two-pointer swap from both ends.*

*Every element is swapped about twice, so it's O(n) time with O(1) extra space."*